In [1]:
# !pip install "vllm>=0.8.5" mcp ddgs smolagents markdownify

In [2]:
!nvidia-smi

Thu Dec  4 23:21:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...    Off |   00000000:01:00.0  On |                  N/A |
| N/A   55C    P4             11W /   55W |     176MiB /   8188MiB |     34%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import typing as tp
import subprocess

# Model setup

**Visit https://openrouter.ai/ to create an account, generate api key and search for models**

In [4]:
PROVIDER = 'OpenRouter' # OpenRouter or vLLM
# PROVIDER = 'vLLM' # OpenRouter or vLLM

In [5]:
from getpass import getpass
if PROVIDER == 'vLLM':
    KEY = "EMPTY"
    HOST = "127.0.0.1"
    PORT = "8999"
    MODEL = "Qwen/Qwen3-4B-Instruct-2507"
    URL = f"http://{HOST}:{PORT}/v1"
else:
    MODEL = "x-ai/grok-4.1-fast"
    KEY = getpass("Your openrouter api key: ")
    URL = f"https://openrouter.ai/api/v1"


Your openrouter api key:  ········


In [6]:
if PROVIDER == 'vLLM':
    vllm_server = subprocess.Popen([
        "python",
        "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--max_model_len", "16384",
        "--host", HOST,
        "--port", PORT,
        "--gpu_memory_utilization", "0.6",
        "--max_num_seqs", "16",
    ])
    #  python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-4B-Instruct-2507 --max_model_len 16384 --host 127.0.0.1 --port 8999 --gpu_memory_utilization 0.6 --max_num_seqs 16 --temperature 0.9

# vllm_server.terminate() — остановить


In [7]:
from openai import OpenAI

client = OpenAI(
  base_url=URL,
  api_key=KEY,
)


response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Hi"}],
)

print(response.choices[0].message)

ChatCompletionMessage(content="Hi! What's up?", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None, reasoning_details=[{'id': 'rs_79d01506-b018-28d2-a3f8-937ae6d0e984', 'format': 'xai-responses-v1', 'index': 0, 'type': 'reasoning.encrypted', 'data': 'WRQlxjwcV49mV3NXk/XB4/4DhKzRkgzS2X+eZS6mdL794RSOoPFLxSd/FN7M6zCykF2pzpE5HueS0KL1+mFa43l6JIsXb7XwjAj2NmfXOp0gn+THyA177mvy98yFszvbi4JYYeG1052ZhDPEWiErQAEr5/G8pzuejjy6YAkwPVVbaxxpO11iuhfeRKTRcCgCWxGsOSXues/N7IbJWUUPHkesI8CnR+luhGWDkGxqrUrXcl2VhIlFlCwwWsxdiLgtQcNnq7b6SfB16BK4l6sA7OTe01KcK4ctpks2zUYluRx93htf7pAC0c+mWhyQO+Gh+3rastX386+OvAwTCtu4bIcVodRP+7d1oBe2Oxr4U3xe26jvBQMmdq+ujBs0I1RGqw/ZVDBTDqQQDdNkxsYmyr2bwUaSaIAd1BOmyVXXlPLciI/F2oFpqQajc9G9+kmSQpSiqeAIlMKfMbQPn/WpeSDoS0GsqZ7Itif4H6WGn9WIQw2EhGUgUC4cauuod4GBwS/4eEKdp0dVwD/6ahiA4dzj8zyj9JBKC9CPXrnGm5HsfoA2lY7DgT+X0jVqUmSuFjz0rDSq9t4NNXnf0TnzGh5DTKo1Km/+CoHonQNrTew8CwQYRdT2PRkGck3NvOTErWzy2cf3VSfziHbqgv/+UjsyVmv9Um+kBdW6WFwvrIyJAa

# How to write tools? How to use MCP?

## Tools as functions

In [8]:
from ddgs import DDGS

def duckduckgo_search(query: str, max_results: int = 5) -> list[dict[str, str]]:
    """
    The function to get internet results for given 'query', not more than 'max_results'.
    """
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=max_results, )
        return list(results)

In [9]:
duckduckgo_search('yandex data school')

[{'title': 'Yandex School of Data Analysis · GitHub',
  'href': 'https://github.com/yandexdataschool',
  'body': 'Yandex School of Data Analysis has 114 repositories available. Follow their code on GitHub.'},
 {'title': 'Academic programs at the Yandex School of Data Analysis',
  'href': 'https://dataschool.yandex.com/dataschool/education',
  'body': 'The Yandex School of Data Analysis provides a systematic education, combining practice with theoretical knowledge . Each subject starts with the basics and culminates at the frontiers of science.'},
 {'title': 'Yandex School of Data Analysis',
  'href': 'https://dataschool.yandex.com/',
  'body': 'The two-year Yandex program was created in 2007 and has become Russia’s leading data analysis program. Courses from the Yandex School of Data Analysis serve as the foundation for Master’s programs at major universities, such as the Higher School of Economics and the Moscow Institute of Physics and Technology.'},
 {'title': 'Yandex school of Data

In [10]:
import re
import requests
from markdownify import markdownify
from requests.exceptions import RequestException
def get_webpage_content(url: str) -> str:
    try:
        response = requests.get(url)
        response.raise_for_status()

        # Convert the HTML content to Markdown
        markdown_content = markdownify(response.text).strip()
        # Remove multiple line breaks
        markdown_content = re.sub(r"\n{3,}", "\n\n", markdown_content)

        return markdown_content

    except RequestException as e:
        return f"Error fetching the webpage: {str(e)}"
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"

In [11]:
# Implement function to get top-5 results with full text
def websearch_full_text(query, top_k=1) -> dict[str, tp.Any]:
    headers = duckduckgo_search(query, top_k)
    urls    = [header['href'] for header in headers]
    return list(map(get_webpage_content, urls))

In [12]:
results = websearch_full_text('star wars', top_k=3)
print(results[2])

StarWars.com | The Official Star Wars Website

Skip Navigation

* [More

  More](/)

Search

Cancel

[My Account](javascript:void(0);)
[Logout](javascript:void(0);)

* [other
  ![](https://lumiere-a.akamaihd.net/v1/images/tiktok-logo-white_dd1a4867.svg?region=0%2C0%2C100%2C100)](https://www.tiktok.com/@starwars)
* [instagram](https://www.instagram.com/starwars/)
* [twitter](https://twitter.com/starwars)
* [facebook](https://www.facebook.com/StarWars)
* [youtube](https://www.youtube.com/user/starwars)
* [other
  ![](https://lumiere-a.akamaihd.net/v1/images/sw_nav_kids_937ed58b.svg?region=0%2C0%2C40%2C15)](https://starwarskids.com/)

Skip Navigation

* [NEWS + FEATURES](https://www.starwars.com/news)
  + [THE LATEST](https://www.starwars.com/news)
  + [THE MANDALORIAN AND GROGU](https://www.starwars.com/news/tag/the-mandalorian-and-grogu)
  + [QUIZZES + POLLS](https://www.starwars.com/news/category/quizzes-+-polls)
  + [BOOKS + COMICS](https://www.starwars.com/news/category/books-+-comic

## Run MCP Server

## Connect MCP client

In [13]:
import asyncio
from typing import Optional
from contextlib import AsyncExitStack

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [14]:
class MCPClient:
    def __init__(self, mcps: list[str]):
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self.mcps = mcps
        self.tools = []

    async def connect_to_server(self, server_script_path: str):
        command = "python"
        server_params = StdioServerParameters(
            command=command,
            args=[server_script_path],
            env=None
        )

        stdio_transport = await self.exit_stack.enter_async_context(stdio_client(server_params))
        self.stdio, self.write = stdio_transport
        self.session = await self.exit_stack.enter_async_context(ClientSession(self.stdio, self.write))
        await self.session.initialize()
        response = await self.session.list_tools()
        self.tools = response.tools

    def list_tools(self):
        print("\nConnected to server with tools:", [tool.name for tool in self.tools])
        return self.tools

    async def call_tool(self, tool_name, args):
        result = await self.session.call_tool(tool_name, args)
        return result



In [15]:
client_mcp = MCPClient('')
await client_mcp.connect_to_server('./ysda_tools.py')
tools = client_mcp.list_tools()


Connected to server with tools: ['add', 'subtract', 'multiply', 'divide', 'vector_add', 'vector_subtract', 'vector_dot', 'vector_elementwise_multiply', 'matrix_add', 'matrix_subtract', 'matrix_multiply', 'matrix_transpose']


In [16]:
tools[0]

Tool(name='add', title=None, description='', inputSchema={'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'addArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'addOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None)

# Tool calls example

In [17]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_youtube_captions",
            "description": "Fetch YouTube captions for a given video ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "video_id": {"type": "string"},
                    "lang": {"type": "string", "default": "en"}
                },
                "required": ["video_id"]
            }
        }
    }
]


In [18]:
import inspect
import typing

PYTHON_TO_JSON = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
    dict: "object",
}

def create_tool_description(func):
    sig = inspect.signature(func)
    doc = inspect.getdoc(func) or ""
    params_schema = {"type": "object", "properties": {}, "required": []}

    for name, param in sig.parameters.items():
        annotation = param.annotation
        if annotation in PYTHON_TO_JSON:
            json_type = PYTHON_TO_JSON[annotation]
        else:
            json_type = "string"  # fallback

        entry = {"type": json_type}

        if param.default is not inspect.Parameter.empty:
            entry["default"] = param.default
        else:
            params_schema["required"].append(name)

        params_schema["properties"][name] = entry

    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": doc,
            "parameters": params_schema,
        },
    }


In [19]:
# Convert python function to openai tool
create_tool_description(duckduckgo_search)

{'type': 'function',
 'function': {'name': 'duckduckgo_search',
  'description': "The function to get internet results for given 'query', not more than 'max_results'.",
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string'},
    'max_results': {'type': 'integer', 'default': 5}},
   'required': ['query']}}}

# ReAct Agent from scratch

In [20]:
instruction = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types:
(1) Search[query], which searches the web for the query and returns url, title and small snippet for 5 relevant pages.
(2) Visit web-page[link], which returns content of the page provided.
(3) Finish[answer], which returns the answer and finishes the task.
"""

final_prompt = instruction + """

You must strictly follow this format when answering:

Thought: <your reasoning about what to do next>
Action 1: <one of Search[...], Visit web-page[...], Finish[...]>
Observation: <result of Action 1>
Thought: <next reasoning>
Action 2: <next action>
Observation: <result of Action 2>
...
When you know the answer, use the next Action index with Finish[answer].

Only one Action is allowed per Thought/Action/Observation block.

--------------------
Example 1
--------------------
Question: Who is the current CEO of OpenAI?

Thought: I should search the web to identify the current CEO of OpenAI.
Action 1: Search[current CEO of OpenAI]
Observation:
1. [https://openai.com/about] "OpenAI Leadership" – Overview of OpenAI’s mission and executive team.
2. [https://en.wikipedia.org/wiki/OpenAI] "OpenAI - Wikipedia" – Leadership info.
3. News articles related to current CEO updates.
4. Etc.

Thought: I should check the official OpenAI site for authoritative info.
Action 2: Visit web-page[https://openai.com/about]
Observation: The page states that the current CEO of OpenAI is Sam Altman.

Thought: I now know the answer.
Action 3: Finish[The current CEO of OpenAI is Sam Altman.]

--------------------
Example 2
--------------------
Question: In which year was the Python programming language first released?

Thought: I should search to find the first public release year of Python.
Action 1: Search[Python programming language first released year]
Observation:
1. [https://www.python.org/doc/essays/history] – Official history.
2. Wikipedia entry about Python.
3. Other historical overviews.

Thought: The official history page is most reliable.
Action 2: Visit web-page[https://www.python.org/doc/essays/history]
Observation: Python was first released in 1991.

Thought: I can now answer.
Action 3: Finish[The Python programming language was first released in 1991.]

--------------------
Example 3
--------------------
Question: Summarize the main features of the latest stable release of Django.

Thought: I need to identify the latest stable version and find its feature summary.
Action 1: Search[latest stable Django release features]
Observation:
1. Official Django download page.
2. Django release notes list.
3. Django 4.2 release notes.
4. Overview articles.

Thought: I should confirm the latest stable version from the official source.
Action 2: Visit web-page[https://www.djangoproject.com/download/]
Observation: The latest stable version is Django 4.2.

Thought: Now I need to read the official 4.2 release notes.
Action 3: Visit web-page[https://docs.djangoproject.com/en/stable/releases/4.2/]
Observation: Notes describe:
- Improved form rendering
- Async support enhancements
- ORM performance improvements
- Security/password updates
- Deprecations/removals

Thought: I can now summarize the answer.
Action 4: Finish[The latest stable release of Django is 4.2. Its main features include improved form rendering and widgets, enhanced async support, ORM performance improvements, updated security features, and several API deprecations/removals.]

--------------------
END OF EXAMPLES
--------------------

When you receive a new question, follow the same indexed Thought → Action N → Observation sequence until you can confidently use Finish[answer].
"""

In [21]:
def llm(prompt, stop=["\n"]):
    response = client.chat.completions.create(
      model=MODEL,
      messages=[{"role": "user", "content": prompt}],
      temperature=0,
      max_tokens=100,
      top_p=1,
      frequency_penalty=0.0,
      presence_penalty=0.0,
      stop=stop
    )
    return response.choices[0].message.content

In [22]:
def parse_action(action: str) -> [str, str]:
    action_type = action.split('[')[0]
    action_body = action[len(action_type) + 1:-1]
    return action_type, action_body

def execute_action(action_type: str, action_body: str) -> str:
    if action_type == 'Search':
        return duckduckgo_search(action_body)[0]['body']
    if action_type == 'Visit web-page':
        return get_webpage_content(action_body)[:100]
    if action_type == 'Finish':
        return action_body
    raise RuntimeError(f'Wrong action: {action_type}.')


def call_react(question, prompt=final_prompt, to_print=True):
    len_initial = len(prompt)
    prompt += "\n" + f"Question: {question}" + "\n"
    n_calls, n_badcalls = 0, 0
    r = None
    for i in range(1, 8):
        n_calls += 1
        # Generate thought
        thought_action = llm(prompt)
        if to_print:
            print(thought_action)
        try:
            # Extract generated action
            thought, action = thought_action.strip().split(f"\nAction {i}: ")
        except Exception as e:
            # Error handling
            if to_print:
                print(e)
            n_badcalls += 1
            continue

        action_type, action_body = parse_action(action)
        # Execute action
        obs = execute_action(action_type, action_body)
        # Create observation
        obs = obs.replace('\\n', '')
        step_str = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {obs}\n"
        
        # Add observation to history
        prompt += step_str
        if action_type == 'Finish':
            break
        
    info = {
        'n_calls': n_calls,
        'n_badcalls': n_badcalls,
        'history': prompt[len_initial:],
    }
    return r, info

In [23]:
question = "What studio released star wars?"
r, info = call_react(question, to_print=False)

print(r)
print()

for k, v in info.items():
    print(k)
    print(v)
    print()

None

n_calls
2

n_badcalls
0

history

Question: What studio released star wars?
Thought 1: Thought: I need to determine which studio released the original Star Wars film (likely referring to the 1977 movie). I'll search the web for accurate information.
Action 1: Search[studio that released original Star Wars 1977]
Observation 1: Star Wars (also known as Star Wars : Episode IV - A New Hope) is a 1977 American epic space opera film written and directed by George Lucas, produced by Lucasfilm Ltd. and released by Twentieth Century-Fox.
Thought 2: Thought: The observation clearly states that the original Star Wars (1977) was produced by Lucasfilm Ltd. and released by Twentieth Century-Fox, which is the distributing studio responsible for the theatrical release. This matches my knowledge, so I can confidently answer.
Action 2: Finish[Twentieth Century-Fox (also known as 20th Century Fox) released the original Star Wars film in 1977.]
Observation 2: Twentieth Century-Fox (also known as 20t

# Multi-agent systems with smolagents

In [24]:
from smolagents import tool


@tool
def visit_webpage(link: str) -> str:
  """
  Visit web-page[link], which returns content of the page provided.
  Args:
    link: link to the webpage
  Returns:
    content of the webpage
  """
  return get_webpage_content(link)

In [25]:
from smolagents import (
    CodeAgent,
    OpenAIModel,
    ToolCallingAgent,
    WebSearchTool,
)


model = OpenAIModel(
    model_id=MODEL,
    api_base=URL,
    api_key=KEY,
)


web_agent = CodeAgent(
    tools=[WebSearchTool(), visit_webpage],
    model=model,
    max_steps=5,
    name="web_search_agent",
    description="Runs web searches for you.",
)

In [26]:
!nvidia-smi

Thu Dec  4 23:21:55 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...    Off |   00000000:01:00.0  On |                  N/A |
| N/A   55C    P4             11W /   55W |     176MiB /   8188MiB |     34%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [27]:
manager_agent = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[web_agent],
)

In [28]:
answer = manager_agent.run(
    "How does ReAct agent works? What metrics were reported by authors?"
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How does ReAct agent works? What metrics were reported by authors?                                              │
│                                                                                                                 │
╰─ OpenAIModel - x-ai/grok-4.1-fast ──────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = web_search_agent(                                                                                       
      task="""Search for the original ReAct paper 'ReAct: Synergizing Reasoning and Acting in Language Models' by  
  Shunyu Yao et al. Provide a detailed summary of how the ReAct agent works, including its core mechanism          
  (reasoning-acting loop with Thought-Action-Observation), pseudocode or steps, and key improvements over          
  baselines. Also extract all reported metrics from the paper's experiments, such as success rates or EM scores    
  on benchmarks like HotpotQA, Fever, ALFWorld, WebShop, etc., for ReAct vs. baselines like Chain-of-Thought       
  (CoT). Include table data if available, and cite sections/pages.""",                                             
      additional_args={}                                                                                           
  )                                                                                                                
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

╭────────────────────────────────────────── New run - web_search_agent ───────────────────────────────────────────╮
│                                                                                                                 │
│ You're a helpful agent named 'web_search_agent'.                                                                │
│ You have been submitted this task by your manager.                                                              │
│ ---                                                                                                             │
│ Task:                                                                                                           │
│ Search for the original ReAct paper 'ReAct: Synergizing Reasoning and Acting in Language Models' by Shunyu Yao  │
│ et al. Provide a detailed summary of how the ReAct agent works, including its core mechanism (reasoning-acting  │
│ loop with Thought-Action-Observation), pseudocode or steps, and key improvements over baselines. Also extract   │
│ all reported metrics from the paper's experiments, such as success rates or EM scores on benchmarks like        │
│ HotpotQA, Fever, ALFWorld, WebShop, etc., for ReAct vs. baselines like Chain-of-Thought (CoT). Include table    │
│ data if available, and cite sections/pages.                                                                     │
│ ---                                                                                                             │
│ You're helping your manager solve a wider task: so make sure to not provide a one-line answer, but give as much │
│ information as possible to give them a clear understanding of the answer.                                       │
│                                                                                                                 │
│ Your final_answer WILL HAVE to contain these parts:                                                             │
│ ### 1. Task outcome (short version):                                                                            │
│ ### 2. Task outcome (extremely detailed version):                                                               │
│ ### 3. Additional context (if relevant):                                                                        │
│                                                                                                                 │
│ Put all these in your final_answer tool, everything that you do not pass as an argument to final_answer will be │
│ lost.                                                                                                           │
│ And even if your task resolution is not successful, please return as much context as possible, so that your     │
│ manager can act upon this feedback.                                                                             │
│                                                                                                                 │
╰─ OpenAIModel - x-ai/grok-4.1-fast ──────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

[Step 1: Duration 36.39 seconds| Input tokens: 2,249 | Output tokens: 525]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("""                                                                                                 
  ### How the ReAct Agent Works                                                                                    
  ReAct (from the paper "ReAct: Synergizing Reasoning and Acting in Language Models" by Shunyu Yao et al.,         
  arXiv:2210.03629, ICLR 2023) is a prompting paradigm for large language models (LLMs, e.g., PaLM-540B or GPT-3)  
  that interleaves **verbal reasoning traces ("Thoughts")** and **task-specific "Actions"** in a **closed-loop     
  interaction** with external environments/tools (e.g., Wikipedia API for QA, game simulators like ALFWorld).      
  This synergizes reasoning (dynamic planning, error correction via Chain-of-Thought-like traces) and acting       
  (grounded execution), outperforming pure reasoning (hallucinations), pure acting (no planning), or               
  training-heavy methods (IL/RL).                                                                                  
                                                                                                                   
  #### Core Mechanism (Algorithm 1, §3.1):                                                                         
  1. **Prompt** the LLM with task context \(C_i\), few-shot examples (0-2 shots), and prior trajectory.            
  2. **Generate Thought \(t_{i+1}\)**: Free-form reasoning (e.g., "I need to find X, so search Y"; decomposes      
  goals, plans, reflects on observations, backtracks).                                                             
  3. **Generate Action \(a_{i+1}\)**: Environment-specific (e.g., `search[Paris]` for HotpotQA; `go(entrance)`     
  for ALFWorld).                                                                                                   
  4. **Observe \(o_{i+1}\)**: Feedback from env (e.g., wiki snippet, game state).                                  
  5. **Update** context: \(C_{i+1} = C_i + t_{i+1} + "Action:" + a_{i+1} + "Observation:" + o_{i+1}\).             
  6. **Repeat** until "Finished" (e.g., final Answer in Thought).                                                  
                                                                                                                   
  **Pseudocode**:                                                                                                  
  ```                                                                                                              
  while not Finished(C_i):                                                                                         
      Thought(t_{i+1})  # Reasoning trace                                                                          
      Action(a_{i+1})   # Task action                                                                              
      o_{i+1} ← Env(a_{i+1})  # External feedback                                                                  
      C_{i+1} ← C_i + t_{i+1} + a_{i+1} + o_{i+1}                                                                  
  Output: Final Answer                                                                                             
  ```                                                                                                              
  - **Synergy**: Thoughts guide actions (plan dynamically); Observations ground thoughts (reduce hallucinations).  
  - **Variants**: Dense (QA: verbose reasoning) vs. sparse (games: minimal).                                       
  - Code/prompts: https://github.com/ysymyth/ReAct.                                                                
                                                                                                                   
  #### Key Improvements:                                 

Final answer: 
### How the ReAct Agent Works
ReAct (from the paper "ReAct: Synergizing Reasoning and Acting in Language Models" by Shunyu Yao et al., 
arXiv:2210.03629, ICLR 2023) is a prompting paradigm for large language models (LLMs, e.g., PaLM-540B or GPT-3) 
that interleaves **verbal reasoning traces ("Thoughts")** and **task-specific "Actions"** in a **closed-loop 
interaction** with external environments/tools (e.g., Wikipedia API for QA, game simulators like ALFWorld). This 
synergizes reasoning (dynamic planning, error correction via Chain-of-Thought-like traces) and acting (grounded 
execution), outperforming pure reasoning (hallucinations), pure acting (no planning), or training-heavy methods 
(IL/RL).

#### Core Mechanism (Algorithm 1, §3.1):
1. **Prompt** the LLM with task context \(C_i\), few-shot examples (0-2 shots), and prior trajectory.
2. **Generate Thought \(t_{i+1}\)**: Free-form reasoning (e.g., "I need to find X, so search Y"; decomposes goals, 
plans, reflects on observations, backtracks).
3. **Generate Action \(a_{i+1}\)**: Environment-specific (e.g., `search[Paris]` for HotpotQA; `go(entrance)` for 
ALFWorld).
4. **Observe \(o_{i+1}\)**: Feedback from env (e.g., wiki snippet, game state).
5. **Update** context: \(C_{i+1} = C_i + t_{i+1} + "Action:" + a_{i+1} + "Observation:" + o_{i+1}\).
6. **Repeat** until "Finished" (e.g., final Answer in Thought).

**Pseudocode**:
```
while not Finished(C_i):
    Thought(t_{i+1})  # Reasoning trace
    Action(a_{i+1})   # Task action
    o_{i+1} ← Env(a_{i+1})  # External feedback
    C_{i+1} ← C_i + t_{i+1} + a_{i+1} + o_{i+1}
Output: Final Answer
```
- **Synergy**: Thoughts guide actions (plan dynamically); Observations ground thoughts (reduce hallucinations).
- **Variants**: Dense (QA: verbose reasoning) vs. sparse (games: minimal).
- Code/prompts: https://github.com/ysymyth/ReAct.

#### Key Improvements:
- vs. CoT: Grounds in external tools (no error propagation).
- vs. Act-only: Adds planning/memory.
- Few-shot > IL/RL (beats 100k+ training data).

### Reported Metrics (Paper Tables 1-6, Figs 6/8/10, §4)
Key results on PaLM-540B (0-2 shot unless noted):

- **HotpotQA** (Multi-hop QA, Wikipedia API, dev EM %; Table 1):
  | Method       | PaLM-540B | PaLM-62B Fine-tune |
  |--------------|-----------|--------------------|
  | CoT          | 27.4      | -                  |
  | Act-only     | 25.7      | -                  |
  | **ReAct**    | **35.1**  | **Best**           |
  | SOTA (sup.)  | 67.5      | -                  |

- **Fever** (Fact Verification, Wikipedia API, dev acc %; Table 1):
  | Method       | PaLM-540B |
  |--------------|-----------|
  | CoT          | 60.9      |
  | Act-only     | 58.9      |
  | **ReAct**    | **64.6**  |
  | SOTA (sup.)  | 89.5      |

- **ALFWorld** (Text Game, 2-shot success %; Table 2):
  | Method       | Success % |
  |--------------|-----------|
  | Act-only     | 45        |
  | **ReAct**    | **71**    |
  | IL (100k)    | 37        |

- **WebShop** (E-comm, 1-shot success %; Table 2):
  | Method       | Success % |
  |--------------|-----------|
  | Act-only     | 30.1      |
  | **ReAct**    | **40**    |
  | IL (90k)     | 29.1      |

Others: ScienceQA (ReAct > CoT); Human eval (ReAct most correct); GPT-3 variants similar gains. Full paper: 
https://react-lm.github.io.

[Step 2: Duration 9.11 seconds| Input tokens: 6,453 | Output tokens: 1,757]

In [29]:
answer

'\n### How the ReAct Agent Works\nReAct (from the paper "ReAct: Synergizing Reasoning and Acting in Language Models" by Shunyu Yao et al., arXiv:2210.03629, ICLR 2023) is a prompting paradigm for large language models (LLMs, e.g., PaLM-540B or GPT-3) that interleaves **verbal reasoning traces ("Thoughts")** and **task-specific "Actions"** in a **closed-loop interaction** with external environments/tools (e.g., Wikipedia API for QA, game simulators like ALFWorld). This synergizes reasoning (dynamic planning, error correction via Chain-of-Thought-like traces) and acting (grounded execution), outperforming pure reasoning (hallucinations), pure acting (no planning), or training-heavy methods (IL/RL).\n\n#### Core Mechanism (Algorithm 1, §3.1):\n1. **Prompt** the LLM with task context \\(C_i\\), few-shot examples (0-2 shots), and prior trajectory.\n2. **Generate Thought \\(t_{i+1}\\)**: Free-form reasoning (e.g., "I need to find X, so search Y"; decomposes goals, plans, reflects on observat

In [30]:
from IPython.display import display, Markdown, Latex
display(Markdown(answer))


### How the ReAct Agent Works
ReAct (from the paper "ReAct: Synergizing Reasoning and Acting in Language Models" by Shunyu Yao et al., arXiv:2210.03629, ICLR 2023) is a prompting paradigm for large language models (LLMs, e.g., PaLM-540B or GPT-3) that interleaves **verbal reasoning traces ("Thoughts")** and **task-specific "Actions"** in a **closed-loop interaction** with external environments/tools (e.g., Wikipedia API for QA, game simulators like ALFWorld). This synergizes reasoning (dynamic planning, error correction via Chain-of-Thought-like traces) and acting (grounded execution), outperforming pure reasoning (hallucinations), pure acting (no planning), or training-heavy methods (IL/RL).

#### Core Mechanism (Algorithm 1, §3.1):
1. **Prompt** the LLM with task context \(C_i\), few-shot examples (0-2 shots), and prior trajectory.
2. **Generate Thought \(t_{i+1}\)**: Free-form reasoning (e.g., "I need to find X, so search Y"; decomposes goals, plans, reflects on observations, backtracks).
3. **Generate Action \(a_{i+1}\)**: Environment-specific (e.g., `search[Paris]` for HotpotQA; `go(entrance)` for ALFWorld).
4. **Observe \(o_{i+1}\)**: Feedback from env (e.g., wiki snippet, game state).
5. **Update** context: \(C_{i+1} = C_i + t_{i+1} + "Action:" + a_{i+1} + "Observation:" + o_{i+1}\).
6. **Repeat** until "Finished" (e.g., final Answer in Thought).

**Pseudocode**:
```
while not Finished(C_i):
    Thought(t_{i+1})  # Reasoning trace
    Action(a_{i+1})   # Task action
    o_{i+1} ← Env(a_{i+1})  # External feedback
    C_{i+1} ← C_i + t_{i+1} + a_{i+1} + o_{i+1}
Output: Final Answer
```
- **Synergy**: Thoughts guide actions (plan dynamically); Observations ground thoughts (reduce hallucinations).
- **Variants**: Dense (QA: verbose reasoning) vs. sparse (games: minimal).
- Code/prompts: https://github.com/ysymyth/ReAct.

#### Key Improvements:
- vs. CoT: Grounds in external tools (no error propagation).
- vs. Act-only: Adds planning/memory.
- Few-shot > IL/RL (beats 100k+ training data).

### Reported Metrics (Paper Tables 1-6, Figs 6/8/10, §4)
Key results on PaLM-540B (0-2 shot unless noted):

- **HotpotQA** (Multi-hop QA, Wikipedia API, dev EM %; Table 1):
  | Method       | PaLM-540B | PaLM-62B Fine-tune |
  |--------------|-----------|--------------------|
  | CoT          | 27.4      | -                  |
  | Act-only     | 25.7      | -                  |
  | **ReAct**    | **35.1**  | **Best**           |
  | SOTA (sup.)  | 67.5      | -                  |

- **Fever** (Fact Verification, Wikipedia API, dev acc %; Table 1):
  | Method       | PaLM-540B |
  |--------------|-----------|
  | CoT          | 60.9      |
  | Act-only     | 58.9      |
  | **ReAct**    | **64.6**  |
  | SOTA (sup.)  | 89.5      |

- **ALFWorld** (Text Game, 2-shot success %; Table 2):
  | Method       | Success % |
  |--------------|-----------|
  | Act-only     | 45        |
  | **ReAct**    | **71**    |
  | IL (100k)    | 37        |

- **WebShop** (E-comm, 1-shot success %; Table 2):
  | Method       | Success % |
  |--------------|-----------|
  | Act-only     | 30.1      |
  | **ReAct**    | **40**    |
  | IL (90k)     | 29.1      |

Others: ScienceQA (ReAct > CoT); Human eval (ReAct most correct); GPT-3 variants similar gains. Full paper: https://react-lm.github.io.
